# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/muska123-web/FlyRank-AI-Internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

I chose supervised classification because the task has a defined target indicating whether content is declining. The predicted probability allows the pages to be ranked from highest to lowest decline risk. I compared simple models first and used Precision@50 because the goal is to prioritize the highest-value pages for review.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
import numpy as np

from sklearn.model_selection import GroupShuffleSplit
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score
)


In [5]:
!git clone https://github.com/muska123-web/FlyRank-AI-Internship.git

Cloning into 'FlyRank-AI-Internship'...
remote: Enumerating objects: 149, done.
remote: Counting objects: 100% (149/149), done.
remote: Compressing objects: 100% (106/106), done.
remote: Total 149 (delta 57), reused 89 (delta 27), pack-reused 0 (from 0)
Receiving objects: 100% (149/149), 1.88 MiB | 9.06 MiB/s, done.
Resolving deltas: 100% (57/57), done.


In [6]:
import os

print(os.path.exists(
    "/content/FlyRank-AI-Internship/data/raw/content_refresh_anonymized.csv"
))

True


In [7]:
df = pd.read_csv(
    "/content/FlyRank-AI-Internship/data/raw/content_refresh_anonymized.csv"
)

print("Shape:", df.shape)
df.head()

Shape: (30000, 44)


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [8]:
print(df.columns.tolist())

['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


In [9]:
print(df["trend_direction"].value_counts())

trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64


In [10]:
df["is_declining_label"] = (
    df["trend_direction"] == "down"
).astype(int)

print(df["is_declining_label"].value_counts())

is_declining_label
1    16262
0    13738
Name: count, dtype: int64


In [11]:
leakage_cols = [
    "impressions_last_30d",
    "clicks_last_30d",
    "sessions_last_30d",
    "impressions_prev_30d",
    "clicks_prev_30d",
    "sessions_prev_30d"
]

drop_cols = [
    "is_declining_label",
    "trend_direction",
    "trend_pct",
    "content_id",
    "client_id",
    *leakage_cols
]

X = df.drop(columns=drop_cols)
y = df["is_declining_label"]

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (30000, 34)
y shape: (30000,)


In [ ]:
print(X.columns.tolist())

In [12]:
categorical_cols = X.select_dtypes(
    include=["object"]
).columns.tolist()

numerical_cols = X.select_dtypes(
    exclude=["object"]
).columns.tolist()

print("Categorical columns:", categorical_cols)
print()
print("Number of categorical columns:", len(categorical_cols))
print()
print("Number of numerical columns:", len(numerical_cols))

Categorical columns: ['competition_level', 'content_type', 'main_intent', 'provider_used', 'model_used', 'age_tier', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'impression_tier', 'position_tier']

Number of categorical columns: 11

Number of numerical columns: 23


In [13]:
numeric_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

categorical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipeline, numerical_cols),
        ("cat", categorical_pipeline, categorical_cols)
    ]
)

2. Split design
Grouped by client? Time-aware? Say why this split is honest for your question.

I used a client-holdout split so that the test set contains clients that were not present in training. This prevents the model from being evaluated on rows from clients it has already seen and provides a more realistic validation of performance on unseen clients.

In [14]:
groups = df["client_id"]

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

print(
    "Train clients:",
    df.iloc[train_idx]["client_id"].nunique()
)

print(
    "Test clients:",
    df.iloc[test_idx]["client_id"].nunique()
)

Train shape: (23837, 34)
Test shape: (6163, 34)
Train clients: 25
Test clients: 7


In [15]:
logistic_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            LogisticRegression(
                random_state=42,
                max_iter=3000
            )
        )
    ]
)

logistic_pipeline

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['search_volume',
                                                   'competition', 'cpc',
                                                   'word_count', 'char_count',
                                                   'impressions_90d',
                                                   'clicks_90d',
                                                   'pageviews_90d',
                                                   'sessions_90d', 'users_90d',
                                                   'engaged_sessions_90d',
                                                   'ai_sessions_90d',
                                                   'scroll_ev...
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['competition_level',
                                                   'content_type',
                                                   'main_intent',
                                                   'provider_used',
                                                   'model_used', 'age_tier',
                                                   'freshness_tier',
                                                   'word_count_tier',
                                                   'char_count_tier',
                                                   'impression_tier',
                                                   'position_tier'])])),
                ('classifier',
                 LogisticRegression(max_iter=3000, random_state=42))])

In [16]:
logistic_pipeline.fit(
    X_train,
    y_train
)

print("Logistic Regression trained successfully!")

Logistic Regression trained successfully!


In [17]:
logistic_pred = logistic_pipeline.predict(X_test)

logistic_prob = logistic_pipeline.predict_proba(
    X_test
)[:, 1]

print("Predictions:", len(logistic_pred))
print("Probabilities:", len(logistic_prob))

Predictions: 6163
Probabilities: 6163


In [18]:
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score
)

logistic_roc_auc = roc_auc_score(y_test, logistic_prob)
logistic_avg_precision = average_precision_score(y_test, logistic_prob)
logistic_precision = precision_score(y_test, logistic_pred)
logistic_recall = recall_score(y_test, logistic_pred)
logistic_f1 = f1_score(y_test, logistic_pred)

print(f"ROC AUC: {logistic_roc_auc:.3f}")
print(f"Average Precision: {logistic_avg_precision:.3f}")
print(f"Precision: {logistic_precision:.3f}")
print(f"Recall: {logistic_recall:.3f}")
print(f"F1 Score: {logistic_f1:.3f}")

ROC AUC: 0.577
Average Precision: 0.569
Precision: 0.561
Recall: 0.652
F1 Score: 0.603


In [19]:
def precision_at_k(y_true, scores, k=50):
    y_true = np.asarray(y_true)
    scores = np.asarray(scores)

    top_k_indices = np.argsort(scores)[::-1][:k]

    return y_true[top_k_indices].mean()


logistic_precision_50 = precision_at_k(
    y_test,
    logistic_prob,
    k=50
)

print(f"Logistic Regression Precision@50: {logistic_precision_50:.3f}")

Logistic Regression Precision@50: 0.640


In [20]:
tree_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            DecisionTreeClassifier(
                random_state=42,
                max_depth=8,
                min_samples_leaf=20
            )
        )
    ]
)

tree_pipeline

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['search_volume',
                                                   'competition', 'cpc',
                                                   'word_count', 'char_count',
                                                   'impressions_90d',
                                                   'clicks_90d',
                                                   'pageviews_90d',
                                                   'sessions_90d', 'users_90d',
                                                   'engaged_sessions_90d',
                                                   'ai_sessions_90d',
                                                   'scroll_ev...
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['competition_level',
                                                   'content_type',
                                                   'main_intent',
                                                   'provider_used',
                                                   'model_used', 'age_tier',
                                                   'freshness_tier',
                                                   'word_count_tier',
                                                   'char_count_tier',
                                                   'impression_tier',
                                                   'position_tier'])])),
                ('classifier',
                 DecisionTreeClassifier(max_depth=8, min_samples_leaf=20,
                                        random_state=42))])

In [21]:
tree_pipeline.fit(
    X_train,
    y_train
)

print("Decision Tree trained successfully!")

Decision Tree trained successfully!


In [22]:
tree_pred = tree_pipeline.predict(X_test)

tree_prob = tree_pipeline.predict_proba(
    X_test
)[:, 1]

print("Predictions:", len(tree_pred))
print("Probabilities:", len(tree_prob))

Predictions: 6163
Probabilities: 6163


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

No. Random Forest performed best among the learned models, but the transparent Week-4 rule remains superior for Precision@50.

In [23]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
tree_roc_auc = roc_auc_score(y_test, tree_prob)
tree_avg_precision = average_precision_score(y_test, tree_prob)
tree_precision = precision_score(y_test, tree_pred)
tree_recall = recall_score(y_test, tree_pred)
tree_f1 = f1_score(y_test, tree_pred)

print(f"ROC AUC: {tree_roc_auc:.3f}")
print(f"Average Precision: {tree_avg_precision:.3f}")
print(f"Precision: {tree_precision:.3f}")
print(f"Recall: {tree_recall:.3f}")
print(f"F1 Score: {tree_f1:.3f}")


ROC AUC: 0.589
Average Precision: 0.574
Precision: 0.558
Recall: 0.707
F1 Score: 0.623


In [24]:
tree_precision_50 = precision_at_k(
    y_test,
    tree_prob,
    k=50
)

print(f"Decision Tree Precision@50: {tree_precision_50:.3f}")

Decision Tree Precision@50: 0.440


In [25]:
forest_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            RandomForestClassifier(
                n_estimators=200,
                random_state=42,
                n_jobs=-1,
                min_samples_leaf=5
            )
        )
    ]
)

forest_pipeline

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['search_volume',
                                                   'competition', 'cpc',
                                                   'word_count', 'char_count',
                                                   'impressions_90d',
                                                   'clicks_90d',
                                                   'pageviews_90d',
                                                   'sessions_90d', 'users_90d',
                                                   'engaged_sessions_90d',
                                                   'ai_sessions_90d',
                                                   'scroll_ev...
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['competition_level',
                                                   'content_type',
                                                   'main_intent',
                                                   'provider_used',
                                                   'model_used', 'age_tier',
                                                   'freshness_tier',
                                                   'word_count_tier',
                                                   'char_count_tier',
                                                   'impression_tier',
                                                   'position_tier'])])),
                ('classifier',
                 RandomForestClassifier(min_samples_leaf=5, n_estimators=200,
                                        n_jobs=-1, random_state=42))])

In [26]:
forest_pipeline.fit(
    X_train,
    y_train
)

print("Random Forest trained successfully!")

Random Forest trained successfully!


In [27]:
forest_pred = forest_pipeline.predict(X_test)

forest_prob = forest_pipeline.predict_proba(
    X_test
)[:, 1]

print("Predictions:", len(forest_pred))
print("Probabilities:", len(forest_prob))

Predictions: 6163
Probabilities: 6163


In [28]:
forest_roc_auc = roc_auc_score(y_test, forest_prob)
forest_avg_precision = average_precision_score(y_test, forest_prob)
forest_precision = precision_score(y_test, forest_pred)
forest_recall = recall_score(y_test, forest_pred)
forest_f1 = f1_score(y_test, forest_pred)

print(f"ROC AUC: {forest_roc_auc:.3f}")
print(f"Average Precision: {forest_avg_precision:.3f}")
print(f"Precision: {forest_precision:.3f}")
print(f"Recall: {forest_recall:.3f}")
print(f"F1 Score: {forest_f1:.3f}")

ROC AUC: 0.611
Average Precision: 0.598
Precision: 0.584
Recall: 0.654
F1 Score: 0.617


In [29]:
forest_precision_50 = precision_at_k(
    y_test,
    forest_prob,
    k=50
)

print(f"Random Forest Precision@50: {forest_precision_50:.3f}")

Random Forest Precision@50: 0.660


In [30]:
baseline_test = df.iloc[test_idx].copy()

baseline_test["baseline_score"] = np.where(
    (
        (baseline_test["avg_position"] <= 10)
        & (baseline_test["impressions_90d"] >= 83)
        & (baseline_test["clicks_90d"] == 0)
    ),
    baseline_test["impressions_90d"],
    0
)

baseline_precision_50 = precision_at_k(
    baseline_test["is_declining_label"].values,
    baseline_test["baseline_score"].values,
    k=50
)

print(f"Week-4 Baseline Precision@50: {baseline_precision_50:.3f}")

Week-4 Baseline Precision@50: 0.840


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [31]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
baseline_top50 = baseline_test.sort_values(
    "baseline_score",
    ascending=False
).head(50)

forest_test = df.iloc[test_idx].copy()
forest_test["model_probability"] = forest_prob

forest_top50 = forest_test.sort_values(
    "model_probability",
    ascending=False
).head(50)

print("Baseline top 50 actual declining:")
print(baseline_top50["is_declining_label"].sum())

print()

print("Random Forest top 50 actual declining:")
print(forest_top50["is_declining_label"].sum())


Baseline top 50 actual declining:
42

Random Forest top 50 actual declining:
33


In [32]:
forest_top50["actual_label"] = (
    forest_top50["trend_direction"] == "down"
).astype(int)

forest_errors = forest_top50[
    forest_top50["actual_label"] == 0
]

print("Random Forest top-50 false positives:", len(forest_errors))

forest_errors[
    [
        "content_id",
        "client_id",
        "model_probability",
        "trend_direction",
        "impressions_90d",
        "clicks_90d",
        "avg_position",
        "content_age_days"
    ]
].head(10)

Random Forest top-50 false positives: 17


,content_id,client_id,model_probability,trend_direction,impressions_90d,clicks_90d,avg_position,content_age_days
22042,content_2ba626fea4d6,client_8527a891e2,0.944787,up,360,0,7.2,275
22526,content_1d0963b56227,client_4e07408562,0.940890,up,3445,3,39.0,280
10080,content_35d63627bf3e,client_8527a891e2,0.928123,stable,1525,0,32.6,238
22461,content_7e3be2e230f5,client_4e07408562,0.921275,stable,909,1,33.4,280
13561,content_685d3fea9361,client_8527a891e2,0.919951,up,134,0,9.3,275
5399,content_6677fd6c4ea5,client_4e07408562,0.910871,stable,1152,2,34.8,280
4050,content_500bd3907331,client_4e07408562,0.909771,stable,4037,4,5.5,230
29456,content_b46c62b14582,client_8527a891e2,0.903864,stable,6240,8,31.8,238
2357,content_8f1409b2674e,client_8527a891e2,0.903515,stable,209,0,20.0,271
12069,content_ff4370afd49c,client_4e07408562,0.898347,stable,1677,3,33.1,280


The Random Forest produced 17 false positives among its top 50 ranked pages. Several high-scoring false positives were actually stable or improving pages. This suggests that the model can interpret strong visibility and performance-related signals as decline risk even when the actual trend is not declining. The result helps explain why the Random Forest does not beat the simpler Week-4 baseline on Precision@50.

In [33]:
results = pd.DataFrame({
    "Model": [
        "Week-4 baseline",
        "Logistic Regression",
        "Decision Tree",
        "Random Forest"
    ],
    "ROC AUC": [
        np.nan,
        logistic_roc_auc,
        tree_roc_auc,
        forest_roc_auc
    ],
    "Average Precision": [
        np.nan,
        logistic_avg_precision,
        tree_avg_precision,
        forest_avg_precision
    ],
    "Precision@50": [
        baseline_precision_50,
        logistic_precision_50,
        tree_precision_50,
        forest_precision_50
    ],
    "Precision": [
        np.nan,
        logistic_precision,
        tree_precision,
        forest_precision
    ],
    "Recall": [
        np.nan,
        logistic_recall,
        tree_recall,
        forest_recall
    ],
    "F1": [
        np.nan,
        logistic_f1,
        tree_f1,
        forest_f1
    ]
})

results

,Model,ROC AUC,Average Precision,Precision@50,Precision,Recall,F1
0,Week-4 baseline,NaN,NaN,0.84,NaN,NaN,NaN
1,Logistic Regression,0.577120,0.568620,0.64,0.561423,0.651635,0.603175
2,Decision Tree,0.588978,0.573612,0.44,0.557644,0.706574,0.623337
3,Random Forest,0.611245,0.598499,0.66,0.584445,0.653858,0.617206


In [34]:
print("Rows:", len(df))
print("Features used:", X.shape[1])
print("Train rows:", len(X_train))
print("Test rows:", len(X_test))
print("Train clients:", df.iloc[train_idx]["client_id"].nunique())
print("Test clients:", df.iloc[test_idx]["client_id"].nunique())

print()
print("Best learned model: Random Forest")
print(f"Random Forest Precision@50: {forest_precision_50:.3f}")
print(f"Week-4 baseline Precision@50: {baseline_precision_50:.3f}")

if forest_precision_50 > baseline_precision_50:
    print("The learned model beats the Week-4 baseline.")
else:
    print("The learned model does not beat the Week-4 baseline.")

Rows: 30000
Features used: 34
Train rows: 23837
Test rows: 6163
Train clients: 25
Test clients: 7

Best learned model: Random Forest
Random Forest Precision@50: 0.660
Week-4 baseline Precision@50: 0.840
The learned model does not beat the Week-4 baseline.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.